# Features disponibles en la lectura actual
Diferencia térmica y potencia se recalculan dentro del pipeline. No se incorporan UDI, Product ID, subtipos de falla, baseline_alert, target ni hash.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from src.datos import DOCS, split
from src.pipeline import make_pipeline, PhysicalFeatures


In [2]:
train, _, meta = split()
display(PhysicalFeatures().transform(train).head())
corr=pd.read_csv(DOCS/'spearman.csv',index_col=0)
display(corr[['machine_failure']])

,product_type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,temperature_delta_k,power_w
0,M,298.1,308.6,1551.0,42.8,0.0,10.5,6951.590560
2,L,298.1,308.5,1498.0,49.4,5.0,10.4,7749.387543
3,L,298.2,308.6,1433.0,39.5,7.0,10.4,5927.504659
6,L,298.1,308.6,1558.0,42.4,14.0,10.5,6917.703247
7,L,298.1,308.6,1527.0,40.2,16.0,10.5,6428.264056


,machine_failure
air_temperature_k,0.086395
process_temperature_k,0.044597
rotational_speed_rpm,-0.163535
torque_nm,0.167836
tool_wear_min,0.095188
temperature_delta_k,-0.101465
power_w,0.153197
machine_failure,1.000000


La EDA muestra acoplamiento entre magnitudes físicas y una asociación de las features con el target. Diferencia térmica = proceso menos aire; potencia = torque por rpm por 2 pi / 60. Ambas existen en inferencia porque se calculan con sensores de la lectura, sin consultar la falla. El tipo se codifica one hot dentro de ColumnTransformer. La asociación no basta para afirmar utilidad; la ablación se ejecuta en el notebook 03, después de las líneas base.

In [3]:
from sklearn.linear_model import LogisticRegression
pipe = make_pipeline(LogisticRegression(random_state=42))
display(pipe)

Pipeline(steps=[('features', PhysicalFeatures()),
                ('prepare',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['air_temperature_k',
                                                   'process_temperature_k',
                                                   'rotational_speed_rpm',
                                                   'torque_nm', 'tool_wear_min',
                                                   'temperature_delta_k',
                                                   'power_w']),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['product_type'])])),
                ('model', LogisticRegression(random_state=42))])

Mediana, moda y escalado se ajustan dentro de cada fold. La lista blanca ignora las columnas de auditoría y etiquetas. quality_tier duplica el tipo y se excluye. El proyecto detecta falla actual; no dispone de labels futuras ni historial por máquina.

## Contrato de variables en inferencia

In [4]:
from src.auditoria_features import contract
import pandas as pd
validacion = contract()
display(pd.DataFrame(validacion.pop("inputs")))
display(validacion)

,feature,unit,missing,invalid
0,product_type,L / M / H,0,0
1,air_temperature_k,K,0,0
2,process_temperature_k,K,0,0
3,rotational_speed_rpm,rpm,0,0
4,torque_nm,Nm,0,0
5,tool_wear_min,min,0,0


{'source': 'snowflake',
 'dataset_sha256': 'dbdb2601fec1714c56141c3b24833f6fb671342cadb68d516238a58cbb9f9e1e',
 'train_rows': 8000,
 'invariant_to_target_and_audit_columns': True,
 'single_reading_equivalent_to_batch': True,
 'derived_from_current_reading': True}